# Geographic and Data Noise Analysis
This notebook visualizes outlet locations on a map of Sri Lanka and detects sophisticated data quality issues like ERP noise and system artifacts.

In [7]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster
import os
import re
import numpy as np

silver_path = '../data/silver/'
df_outlets = pd.read_parquet(os.path.join(silver_path, 'dim_outlets.parquet'))
df_trans = pd.read_parquet(os.path.join(silver_path, 'fact_transactions.parquet'))

## 1. Geographic Visualization
Visualizing the distribution of outlets across Sri Lanka.

In [8]:
# Create a base map centered on Sri Lanka
sl_map = folium.Map(location=[7.8731, 80.7718], zoom_start=7)

# Add a marker cluster for performance
marker_cluster = MarkerCluster().add_to(sl_map)

# Sample data for visualization (to keep the map responsive)
sample_outlets = df_outlets.sample(n=1000) if len(df_outlets) > 1000 else df_outlets

for idx, row in sample_outlets.iterrows():
    folium.Marker(
        location=[row['Latitude'], row['Longitude']],
        popup=f"ID: {row['Outlet_ID']}<br>Type: {row['Outlet_Type']}",
    ).add_to(marker_cluster)

sl_map

## 2. Legacy ERP Noise Check
Detecting non-standard patterns in IDs that often stem from old ERP systems.

In [9]:
def check_id_noise(df, col):
    # Check for lowercase (assuming standard is uppercase)
    lowercase_noise = df[df[col].str.contains(r'[a-z]')][col].unique()
    
    # Check for trailing/leading spaces
    space_noise = df[df[col].str.contains(r'^\s|\s$')][col].unique()
    
    # Check for non-alphanumeric (excluding underscores)
    symbol_noise = df[df[col].str.contains(r'[^A-Z0-9_]')][col].unique()
    
    print(f"--- ID Noise Report for {col} ---")
    print(f"Lowercase noise: {len(lowercase_noise)}")
    print(f"Whitespace noise: {len(space_noise)}")
    print(f"Symbol/Special char noise: {len(symbol_noise)}")
    return lowercase_noise, space_noise, symbol_noise

out_noise = check_id_noise(df_outlets, 'Outlet_ID')
dist_noise = check_id_noise(df_trans, 'Distributor_ID')

--- ID Noise Report for Outlet_ID ---
Lowercase noise: 0
Whitespace noise: 0
Symbol/Special char noise: 0
--- ID Noise Report for Distributor_ID ---
Lowercase noise: 0
Whitespace noise: 0
Symbol/Special char noise: 0


## 3. Ghost Entries Check
Identifying 'Zombie' outlets that exist in metadata but have no real economic activity.

In [10]:
# Total activity per outlet
outlet_activity = df_trans.groupby('Outlet_ID')['Total_Bill_Value'].sum().reset_index()

# Ghost outlets: Total sales < 1.0 (threshold for negligible activity)
ghost_outlets = outlet_activity[outlet_activity['Total_Bill_Value'] < 1.0]
print(f"Number of Ghost Outlets: {len(ghost_outlets)}")

if not ghost_outlets.empty:
    print("Sample Ghost IDs:", ghost_outlets['Outlet_ID'].head().tolist())

Number of Ghost Outlets: 0


## 4. Batch Writes Analysis
Detecting suspicious clusters of transactions that may have been written in automated batches.

In [11]:
print("--- Batch Writes Report ---")

# 1. High frequency of identical Total_Bill_Value within the same (Distributor, Month)
batch_candidates = df_trans.groupby(['Distributor_ID', 'Year', 'Month', 'Total_Bill_Value']).size().reset_index(name='count')
suspicious_batches = batch_candidates[batch_candidates['count'] > 50] # Threshold: >50 identical sales in one month by one distributor

print(f"Number of suspicious identical-value clusters (>50 records): {len(suspicious_batches)}")
if not suspicious_batches.empty:
    display(suspicious_batches.sort_values('count', ascending=False).head(10))

# 2. Monthly transaction count spikes per distributor
dist_monthly_counts = df_trans.groupby(['Distributor_ID', 'Year', 'Month']).size().reset_index(name='tx_count')
mean_count = dist_monthly_counts['tx_count'].mean()
std_count = dist_monthly_counts['tx_count'].std()

tx_spikes = dist_monthly_counts[dist_monthly_counts['tx_count'] > (mean_count + 3 * std_count)]
print(f"\nNumber of months with transaction spikes (>3 STD from mean): {len(tx_spikes)}")

--- Batch Writes Report ---
Number of suspicious identical-value clusters (>50 records): 0

Number of months with transaction spikes (>3 STD from mean): 0


## 5. Data Decay Check
Detecting outlets that 'died' (stopped reporting) during the observation period.

In [12]:
df_trans['Date'] = pd.to_datetime(df_trans[['Year', 'Month']].assign(day=1))
max_date = df_trans['Date'].max()

outlet_last_seen = df_trans.groupby('Outlet_ID')['Date'].max().reset_index()
decayed_outlets = outlet_last_seen[outlet_last_seen['Date'] < (max_date - pd.DateOffset(months=6))]

print(f"Number of outlets inactive for >6 months (Data Decay): {len(decayed_outlets)}")

Number of outlets inactive for >6 months (Data Decay): 95


## 6. System Artifacts Check
Detecting 'Magic Numbers' or hardcoded system defaults.

In [13]:
magic_numbers = [9999, 1111, 1234, 0, -1]

print("--- System Artifact Report ---")
for num in magic_numbers:
    hits = df_trans[df_trans['Total_Bill_Value'] == num]
    if not hits.empty:
        print(f"Magic Number {num} found {len(hits)} times.")

# Check for exact repeating bill values across different distributors
top_values = df_trans['Total_Bill_Value'].value_counts().head(10)
print("\nTop repeating bill values across all transactions:")
print(top_values)

--- System Artifact Report ---

Top repeating bill values across all transactions:
Total_Bill_Value
2177.632359     1
7244.084814     1
13959.108787    1
15641.548773    1
15525.158656    1
3714.516869     1
1261.701912     1
14613.706566    1
1985.480389     1
10078.206589    1
Name: count, dtype: int64
